# escape.stream_new — local test stream example

Demonstrates the **`Stream`** class against a synthetic local bsread stream  
(no beamline connection needed).

The `Stream` API mirrors `escape.Array`:

| Array | Stream equivalent |
|---|---|
| `Array("ch", ...)` | `Stream("ch", ew)` |
| `arr[bool_mask]` | `stream[mask_stream]` |
| `arr.digitize(bins).categorize(other)` | `t.digitize(bins).categorize(ratio)` |
| `arr.plot()` | `stream.plot_med()` / `plot_hist()` / `plot_corr()` |

Test stream channels (~100 Hz):

| Channel | Description |
|---|---|
| `i0` | Incoming photon flux |
| `i` | Transmitted signal |
| `t` | Pump-probe delay (ps) |
| `pump_on` | Laser on (1) / off (0) |
| `drift` | Slow beam drift |

**Prerequisite:** `%matplotlib widget` (ipympl) for live-updating plots.

In [ ]:
%matplotlib widget

## 1 — Imports

In [ ]:
import sys, time
sys.path.insert(0, '/das/home/lemke_h/mypy/escape-fel/')

import numpy as np
import matplotlib.pyplot as plt

from escape.stream_new import (
    Stream, StreamContext, EventWorker,
    LocalEventHandler,          # legacy bsread-direct handler
    DataHubLocalEventHandler,   # datahub wrapper (default — same result, future-proof)
    DataHubEventHandler,        # live SwissFEL dispatcher / Redis
    MultiSourceEventHandler,    # merge channels from multiple backends
    TestStream,
)

## 2 — Start the synthetic test stream

`TestStream` launches `createStream()` in a child process, binding a bsread Sender
to `localhost:9999`.  It broadcasts all channels until stopped.

In [ ]:
ts = TestStream(port=9999, interval=0.01)   # ~100 Hz
ts.start()
time.sleep(0.5)   # give the sender a moment to bind
ts

## 3 — Create Streams

Short constructor: `Stream("channel_name", ew, unit="...")` — mirrors Array.

`DataHubLocalEventHandler` connects directly to `localhost:9999` using the datahub
abstraction layer.  It replaces the legacy `LocalEventHandler` but has the same
interface.  All channels broadcast by the sender are received automatically.

**For live SwissFEL data**, swap the handler:
```python
ew = EventWorker(DataHubEventHandler(backend='bsread'))   # bsread via dispatcher
ew = EventWorker(DataHubEventHandler(backend='redis'))    # Redis/Dragonfly
```

In [ ]:
# DataHubLocalEventHandler: datahub wrapper for local/test bsread streams.
# Channels=[] → receive all; no restart needed when adding new Streams.
ew = EventWorker(eventHandler=DataHubLocalEventHandler(host='localhost', port=9999), make_default=True)

i0    = Stream('i0',      ew, unit='a.u.')
i     = Stream('i',       ew, unit='a.u.')
t     = Stream('t',       ew, unit='ps')
pump  = Stream('pump_on', ew, unit='bool')
drift = Stream('drift',   ew, unit='a.u.')

print('Streams created:', i0, i, t, pump, drift, sep='\n  ')

## 4 — Arithmetic and filtering

Operator overloading returns new derived Streams — exactly like `escape.Array`.

`~pump` gives logical NOT (True when laser is off), so `i[~pump]` accumulates
only pump-off shots.

In [ ]:
ratio  = i / i0         # per-shot ratio — derived ProcSource-backed Stream
i_on   = i[pump]        # pump-on shots only  (filter by boolean stream)
i_off  = i[~pump]       # pump-off shots  (~pump = logical NOT)

print('ratio :', ratio)
print('i_on  :', i_on)
print('i_off :', i_off)

## 5 — Accumulate and inspect

`accumulate(True)` registers the channel with the EventWorker and fills
the circular buffer (default 1000 events per scan step).

In [ ]:
for stream in [i0, i, t, pump, drift, ratio, i_on, i_off]:
    stream.accumulate(True)

time.sleep(3)   # collect ~300 events

print(f'i0    : {len(i0):4d} events')
print(f'ratio : {len(ratio):4d} events')
print(f'i_on  : {len(i_on):4d} events (pump-on only)')
print(f'i_off : {len(i_off):4d} events (pump-off only)')
print(f'i_on + i_off ≈ i0: {len(i_on) + len(i_off)} vs {len(i0)}')

## 6 — Live value-distribution histogram

`plot_hist()` auto-detects no-scan mode and opens a ValueHistPlot (value
distribution, not step counts).

In [ ]:
fig_h, ax_h = plt.subplots(figsize=(6, 3))
hp = i0.plot_hist(update=0.5, axes=ax_h, n_bins=40)

## 7 — Live correlation scatter plot

Shows the last 400 events of `i` vs `i0`, matched by pulse ID.

In [ ]:
fig_c, ax_c = plt.subplots(figsize=(5, 5))
cp = i.plot_corr(i0, Npoints=400, update=0.5, axes=ax_c)

## 8 — Binning by delay: `t.digitize(bins).categorize(ratio)`

Mirrors `escape.Array.digitize().categorize()`.  The test stream broadcasts
random delay values t ∈ [-1, 1] ps; here we bin the i/i0 ratio by delay.

In [ ]:
bins = np.linspace(-1.0, 1.0, 21)   # 20 bins across the delay range

ratio_vs_t = t.digitize(bins).categorize(ratio)
print(ratio_vs_t)

ratio_vs_t.accumulate(True)
time.sleep(4)

print(f'Events per bin: {ratio_vs_t.lens()}')

### Live median plot of the binned ratio

In [ ]:
fig_r, ax_r = plt.subplots(figsize=(7, 3))
fig_r.suptitle('i/i0 vs delay  (live)')
mp_r = ratio_vs_t.plot_med(update=0.5, axes=ax_r)

### Pump-on only

In [ ]:
ratio_on     = i_on / i0
ratio_on_vst = t.digitize(bins).categorize(ratio_on)
ratio_on_vst.accumulate(True)

fig_on, ax_on = plt.subplots(figsize=(7, 3))
fig_on.suptitle('pump-on i/i0 vs delay  (live)')
mp_on = ratio_on_vst.plot_med(update=0.5, axes=ax_on)

## 9 — StreamContext: managed acquisition lifetime

Accumulate for a fixed duration or tie acquisition to a figure window.

In [ ]:
with StreamContext(drift):
    print('Accumulating drift for 3 s…')
    time.sleep(3)

drift_arr = np.array(drift.data[0])
print(f'Collected {len(drift_arr)} drift events,  mean = {drift_arr.mean():.4f}')

In [ ]:
fig_d, ax_d = plt.subplots(figsize=(6, 3))
fig_d.suptitle('drift  (close window to stop)')

ctx = StreamContext(drift)
ctx.tie_to_figure(fig_d)
ctx.start()

mp_d = drift.plot_med(update=0.5, axes=ax_d)
print('Close the figure above to automatically stop drift accumulation.')

## 10 — Raw data access via slice syntax

`stream[-n:]` returns the last *n* accumulated events as a numpy array.

In [ ]:
last_100_i0 = i0[-100:]      # last 100 events
print(f'i0 last 100:  mean={last_100_i0.mean():.4f}  std={last_100_i0.std():.4f}')

arr_on  = np.array(i_on.data[0])
arr_off = np.array(i_off.data[0])
print(f'pump-on  i:  mean={arr_on.mean():.4f}  n={len(arr_on)}')
print(f'pump-off i:  mean={arr_off.mean():.4f}  n={len(arr_off)}')

## 10b — Multi-source merging

`MultiSourceEventHandler` fuses channels from **two independent bsread senders**
into a single event stream, aligned by pulse_id.

This mirrors the real use-case: some channels on the SwissFEL bsread dispatcher
and others on a second backend (Redis, special host, or another beamline segment).

Here we start a minimal second sender on port 9998 that broadcasts one extra channel
(`extra_signal`), then create an `EventWorker` that merges both streams.

In [ ]:
from multiprocessing import Process

def _secondary_stream(port=9998, interval=0.02):
    """Minimal bsread sender broadcasting a single 'extra_signal' channel."""
    import time
    from bsread.sender import Sender
    sender = Sender(port=port)
    sender.add_channel_from_value('extra_signal', 1.0)
    sender.open()
    pulse_id = 1
    try:
        while True:
            sender.send(float(pulse_id * 0.01), pulse_id=pulse_id, check_data=False)
            pulse_id += 1
            time.sleep(interval)
    except Exception:
        pass
    finally:
        sender.close()

# Start secondary stream on port 9998
ts2 = Process(target=_secondary_stream, args=(9998, 0.02), daemon=True)
ts2.start()
time.sleep(0.5)
print('Secondary stream started on port 9998.')

# Create a merged EventWorker: port 9999 (i0/i/t/pump) + port 9998 (extra_signal)
ew_multi = EventWorker(
    MultiSourceEventHandler(
        DataHubLocalEventHandler(host='localhost', port=9999),
        DataHubLocalEventHandler(host='localhost', port=9998),
        timeout_pulses=30,    # forward partial events after 30 missed pulse IDs
    ),
    make_default=False,
)

# Both sources available in the same event dict
i0_m     = Stream('i0',           ew_multi, unit='a.u.')
extra    = Stream('extra_signal', ew_multi, unit='a.u.')

i0_m.accumulate(True)
extra.accumulate(True)

time.sleep(4)

print(f'i0    events: {len(i0_m)}  (from port 9999)')
print(f'extra events: {len(extra)}  (from port 9998)')
print(f'Extra signal mean: {np.array(extra.data[0]).mean():.4f}')

# Both streams should have very similar counts (aligned by pulse_id)
print('Multi-source merge: OK' if len(i0_m) > 50 and len(extra) > 50 else 'WARNING: low event count')

## 11 — Stop everything

In [ ]:
# Stop live-plot update threads
for plot_obj in [hp, cp, mp_r, mp_on, mp_d]:
    try:
        plot_obj.stop()
    except Exception:
        pass

# Stop all channel accumulation
for stream in [i0, i, t, pump, drift, ratio, i_on, i_off,
               ratio_vs_t, ratio_on_vst, i0_m, extra]:
    try:
        stream.accumulate(False)
    except Exception:
        pass

ew.stopEventLoop()
try:
    ew_multi.stopEventLoop()
    ts2.terminate()
    ts2.join(timeout=2)
except Exception:
    pass

ts.stop()
print('Done.')